# Lab 3 — When things fail (EP08)

From the theory video (§18.7): retries with **exponential backoff + jitter**, a **fallback model**, **loop detection** and **graceful failure**.
In EP06 Lab 1 we retried by waiting 1 then 2 seconds — this lab uses the textbook's full formula

**What you'll learn**
1. The backoff formula (Equation 18.14), and retrying an API that answers 429
2. Why jitter: ten agents rate-limited at the same moment
3. A fallback model: the main model is unavailable → switch to a backup
4. An agent stuck calling the same tool again and again → caught with a hash in a sliding window (Equation 18.15), then a graceful stop

The API answering 429, the main model's outage and the PDF that never finishes converting are **simulated failures for teaching**.
The model's repeated tool calls and every answer come from real runs.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §18.7 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | the same key as Lab 1 (this lab costs under 1 US cent) |
| Packages | `openai`, `python-dotenv` |

How to add the key, and why we chose this model and provider, are in **Lab 1**

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
import hashlib, json, os, random, time
from collections import deque
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["makora"], "allow_fallbacks": False}}
USAGE = {"calls": 0, "in": 0, "out": 0}

## 1. Exponential backoff + jitter (Equation 18.14)

$$t_k = \min\left(2^k \cdot t_0 + U(0, t_0),\ t_{max}\right)$$

k = which retry (from 0) · $t_0$ = the base delay · $U(0, t_0)$ = a random amount (jitter) · $t_{max}$ = the ceiling
· this lab uses $t_0$ = 0.5 seconds so it runs quickly

In [ ]:
def backoff_delay(k, t0=0.5, t_max=8.0, jitter=True):
    extra = random.uniform(0, t0) if jitter else 0
    return min(2 ** k * t0 + extra, t_max)

In [ ]:
class RateLimited(Exception):
    pass

TRANSIENT = (RateLimited, ConnectionError, TimeoutError)

def call_with_retry(fn, retries=4):
    for k in range(retries + 1):
        try:
            return fn()
        except TRANSIENT as e:
            if k == retries:
                raise
            wait = backoff_delay(k)
            print(f"  retry {k}: {type(e).__name__} → wait {wait:.2f} s")
            time.sleep(wait)

In [ ]:
BUSY = {"left": 3}

def search_api():
    if BUSY["left"] > 0:
        BUSY["left"] -= 1
        raise RateLimited("429 too many requests")
    return ["Informer (2021)", "PatchTST (2023)", "TimesFM (2024)"]

t0 = time.time()
print(call_with_retry(search_api), f"after {time.time() - t0:.1f} s")

## 2. Why jitter

Ten agents share one key and all get a 429 **at the same moment**, time 0 — when does each come back to retry?

In [ ]:
def retry_times(jitter, agents=10, retries=3):
    times = []
    for _ in range(agents):
        t, mine = 0.0, []
        for k in range(retries):
            t += backoff_delay(k, jitter=jitter)
            mine.append(round(t, 2))
        times.append(mine)
    return times

random.seed(0)
for jitter in (False, True):
    first = sorted(t[0] for t in retry_times(jitter))
    print(f"jitter={jitter!s:5}  1st retry of the 10 agents: {first}")

## 3. A fallback model

Textbook §18.7.1: the main model is unavailable → switch to a **backup model**, maybe less capable but available

| | model | provider |
|---|---|---|
| Main | Gemma 4 26B | Makora |
| Backup | Gemma 3 12B (smaller, cheaper) | DeepInfra |

The main model's outage is simulated (`OUTAGE`); the backup model's answer comes from a real run

In [ ]:
MODELS = [
    ("google/gemma-4-26b-a4b-it", {"order": ["makora"], "allow_fallbacks": False}),         # main
    ("google/gemma-3-12b-it", {"order": ["deepinfra/bf16"], "allow_fallbacks": False}),     # backup
]
OUTAGE = set()   # models we pretend are down

def ask_model(model, provider, messages):
    if model in OUTAGE:
        raise ConnectionError(f"503 {model} unavailable")
    reply = client.chat.completions.create(model=model, messages=messages, max_tokens=200,
                                           extra_body={"provider": provider})
    USAGE["calls"] += 1; USAGE["in"] += reply.usage.prompt_tokens; USAGE["out"] += reply.usage.completion_tokens
    return reply.choices[0].message.content

def ask_with_fallback(messages):
    for model, provider in MODELS:
        try:
            return model, call_with_retry(lambda: ask_model(model, provider, messages), retries=2)
        except TRANSIENT as e:
            print(f"  ✖ {model}: {e} → next model")
    raise RuntimeError("every model failed")

In [ ]:
OUTAGE.add("google/gemma-4-26b-a4b-it")
model, answer = ask_with_fallback([{"role": "user", "content": "In one sentence: what is PatchTST?"}])
print(f"\nanswered by {model}:\n{answer}")
OUTAGE.clear()

## 4. An agent stuck in a loop

`get_full_text` simulates a tool whose PDF conversion never finishes: it answers `"processing … Call again"` every time.
An agent with nothing to stop it calls the same request again and again until `max_steps` (our last-resort guard since EP03)

In [ ]:
def get_full_text(paper_id):
    return {"status": "processing", "message": "The PDF is still being converted. Call again to get the text."}

TOOLS = [{"type": "function", "function": {
    "name": "get_full_text",
    "description": "Get the full text of a paper by id. The PDF is converted on demand; if it is not ready yet, call again.",
    "parameters": {"type": "object", "properties": {"paper_id": {"type": "string"}}, "required": ["paper_id"]}}}]

def chat(messages):
    reply = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, extra_body=EXTRA)
    USAGE["calls"] += 1; USAGE["in"] += reply.usage.prompt_tokens; USAGE["out"] += reply.usage.completion_tokens
    return reply.choices[0].message

In [ ]:
def run_agent(question, guard=None, max_steps=12):
    messages = [{"role": "system", "content": "You are a research assistant. Use the tools to answer."},
                {"role": "user", "content": question}]
    for step in range(max_steps):
        reply = chat(messages)
        if not reply.tool_calls:
            return reply.content
        messages.append(reply.model_dump(exclude_none=True))
        for call in reply.tool_calls:
            args = json.loads(call.function.arguments)
            print(f"  step {step}: {call.function.name}({args})")
            if guard and guard.is_loop(call.function.name, args):
                return graceful_failure(guard, messages)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(get_full_text(**args))})
    return "stopped: max_steps"

In [ ]:
USAGE["calls"] = 0
print(run_agent("Get the full text of paper P3 and quote its first sentence."))
print(f"\nmodel calls: {USAGE['calls']}")

In [ ]:
class LoopGuard:
    def __init__(self, window=5, k=3):
        self.recent = deque(maxlen=window)   # the sliding window W
        self.k = k

    def is_loop(self, tool, args):
        h = hashlib.sha256(f"{tool}:{json.dumps(args, sort_keys=True)}".encode()).hexdigest()[:8]
        self.recent.append(h)
        return self.recent.count(h) >= self.k

In [ ]:
def graceful_failure(guard, messages):
    with open("stuck_task.json", "w") as f:
        json.dump(messages, f)
    return ("✅ Done: asked for the full text of P3 (the request is valid).\n"
            f"✖ Why: get_full_text answered 'processing' {guard.k} times in a row — I stopped repeating it.\n"
            "➜ Next: try again in a few minutes, or use the abstract instead.\n"
            "💾 State saved to stuck_task.json — the task can resume from here.")

In [ ]:
USAGE["calls"] = 0
print(run_agent("Get the full text of paper P3 and quote its first sentence.", guard=LoopGuard()))
print(f"\nmodel calls: {USAGE['calls']}")

In [ ]:
cost = USAGE["in"] / 1e6 * 0.09 + USAGE["out"] / 1e6 * 0.28
print(f"≈ ${cost:.4f} for the whole lab (Makora prices; the backup model costs less)")

## Summary

| What the harness does | What we saw in the lab |
|---|---|
| Exponential backoff | Waits of ~0.5 → ~1 → ~2 seconds, then the fourth try succeeds |
| Jitter | Without: all ten agents come back at 0.5 s together · with: spread across 0.5–1 s |
| Fallback model | Main model down → tried 3 times → Gemma 3 12B answers instead |
| Loop detection | No guard: the same request repeated round after round · with the guard: stopped at 3 |
| Graceful failure | What was done · why it stopped · what to do next · state saved |

**Exercises**
1. Try LoopGuard on a `search_papers` that answers "No results" every time — the model rewrites the query each round, so the hash never repeats. How would you catch that kind of loop? (Hint: the textbook's third method — check whether **the state changes**, instead of the arguments)
2. Put both models in `OUTAGE` — what does `ask_with_fallback` do, and how should the harness tell the user?

**Next — Lab 4:** the cost of a task, and watching prompt caching really work through `cached_tokens`